In [1]:
from bs4 import BeautifulSoup

In [2]:
# eng.lsm.lv sits behind Cloudflare, so a plain requests.get() gets a 403
# "Just a moment..." block (unlike el_pais / lrt_lt which work directly).
# So we render the live page with Playwright (stealth) to get past the
# challenge, then hand the HTML to BeautifulSoup.
import sys
import asyncio
import threading

from playwright.async_api import async_playwright
from playwright_stealth import Stealth

URL = "https://eng.lsm.lv/"


async def fetch_html():
    async with Stealth().use_async(async_playwright()) as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        # "commit" returns as soon as the response arrives so a Cloudflare
        # interstitial can't stall navigation; then we wait for the real
        # content selector to appear after the challenge clears.
        await page.goto(URL, wait_until="commit", timeout=40000)
        await page.wait_for_selector(".list-article", timeout=40000)
        html = await page.content()
        await browser.close()
        return html


# Playwright launches a browser subprocess, which on Windows needs a Proactor
# event loop -- but Jupyter's own loop is a Selector loop. So we run the fetch
# on its own thread with a fresh loop to avoid clashing with the kernel.
result = {}


def runner():
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    result["html"] = loop.run_until_complete(fetch_html())
    loop.close()


thread = threading.Thread(target=runner)
thread.start()
thread.join()

doc = BeautifulSoup(result["html"], "html.parser")

In [3]:
# What is the selector for the headlines WITH THE LINKS?
# Works from right to left
# article means an <article> tag
# .list-article means something with the class "list-article"
# so this means <article> tags with the class list-article
items = doc.select("article.list-article")
len(items)

56

In [4]:
# Take a look at one item to see which DOM elements hold what we want:
#   <time>   -> the publish time
#   <a>      -> the headline text + the link (href)
items[0]

<article class="list-article"><time>20:35</time><div><a href="/article/culture/sport/12.07.2026-watch-live-latvia-versus-ivory-coast-u-17-womens-basketball.a654768/" target="_self">WATCH LIVE: Latvia versus Ivory Coast U-17 women's basketball<i class="icon-play"></i></a></div></article>

In [5]:
# We love a good CSV file
# we probably want one with a HEADLINE column and also a URL column
#
# to make a CSV we need a LIST of DICTIONARIES
# 1. Make an empty list called articles
# 2. Add our data dictionary to it each time we loop
articles = []
for item in items:
    link = item.select_one("a")
    if not link:
        continue
    time = item.select_one("time")
    article = {
        "url": "https://eng.lsm.lv" + link["href"],
        "headline": link.get_text(strip=True),
        "time": time.get_text(strip=True) if time else None,
    }
    articles.append(article)
len(articles)

56

In [6]:
import pandas as pd

# Just feed the list of dictionaries to pandas
# and it makes us a nice beautiful dataframe
df = pd.DataFrame(articles)
df.head()

,url,headline,time
0,https://eng.lsm.lv/article/culture/sport/12.07...,WATCH LIVE: Latvia versus Ivory Coast U-17 wom...,20:35
1,https://eng.lsm.lv/article/weather/weather/12....,Storm warning for eastern Latvia on Sunday eve...,17:38
2,https://eng.lsm.lv/article/economy/transport/1...,Death on railway line in Rīga,15:07
3,https://eng.lsm.lv/article/features/features/1...,The Deep Roots of Resilience: what a crown tau...,12:00
4,https://eng.lsm.lv/article/economy/employment/...,4.8% of Latvia's jobs are in culture,10:42


In [7]:
import os

# Try to create a folder for the data
# and if it exists DON'T THROW AN ERROR
os.makedirs("data/lsm_lv", exist_ok=True)

In [8]:
from datetime import datetime

# Timestamp down to the second so each scrape gets its own file
date_string = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")
filepath = f"data/lsm_lv/{date_string}.csv"

df.to_csv(filepath, index=False)

In [ ]:
# Append to an always-updated file, deduplicated by url.

# add columns for when we scraped this
df['scrape_date'] = datetime.now().strftime("%Y-%m-%d")
df['scrape_datetime'] = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")

# open the always-updated file if it exists, else start blank
try:
    existing_df = pd.read_csv("lsm_latvia-always-updated.csv")
except:
    existing_df = pd.DataFrame([])

# OLD first, then NEW, so keep='first' preserves the earliest scrape date
combined = pd.concat([existing_df, df], ignore_index=True)
print("Before dropping duplicates:", len(combined))

# dedup on url ONLY -> only genuinely new articles get added to the file
combined = combined.drop_duplicates(subset=['url'], keep='first')
print("After dropping duplicates: ", len(combined))

combined.to_csv("lsm_latvia-always-updated.csv", index=False)
combined.head()